# Modelagem — Baseline

Regressão logística com a variável `vertical` agrupada em três faixas de risco (definidas no notebook 1) e codificada via WoE.

In [1]:
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

sys.path.append("../")

from utils.metrics import metrics_by_split
from utils.woe import WoEEncoder

import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/nuvem_fraud_dataset_pos_eda.csv", sep=";", index_col=0)
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df.head()

,created_at,txn_id,customer_id,amount,vertical,is_fraud,txns_prev_5min,txns_prev_15min,txns_prev_30min,txns_prev_1h,txns_prev_6h,txns_prev_24h,amount_to_customer_mean_ratio_30d,amount_to_customer_median_ratio_30d,amount_zscore_customer_30d
0,2019-01-01 00:00:44+00:00,0,122,107.23,grocery_pos,0,0,0,0,0,0,0,NaN,NaN,NaN
1,2019-01-01 00:00:51+00:00,1,63,220.11,entertainment,0,0,0,0,0,0,0,NaN,NaN,NaN
2,2019-01-01 00:07:27+00:00,2,23,96.29,grocery_pos,0,0,0,0,0,0,0,NaN,NaN,NaN
3,2019-01-01 00:09:03+00:00,3,83,7.77,shopping_pos,0,0,0,0,0,0,0,NaN,NaN,NaN
4,2019-01-01 00:21:32+00:00,4,81,6.85,misc_pos,0,0,0,0,0,0,0,NaN,NaN,NaN


## Agrupamento da vertical

Mesma recategorização do notebook 1: alto, médio e baixo risco.

In [3]:
vertical_groups = {
    "vertical_high_risk": ["shopping_net", "grocery_pos", "misc_net"],
    "vertical_medium_risk": ["shopping_pos", "gas_transport", "travel", "misc_pos"],
    "vertical_low_risk": [
        "grocery_net", "entertainment", "personal_care", "kids_pets",
        "food_dining", "health_fitness", "home",
    ],
}

vertical_to_group = {
    vertical: group
    for group, verticals in vertical_groups.items()
    for vertical in verticals
}

df["vertical_group"] = df["vertical"].map(vertical_to_group)
assert df["vertical_group"].notna().all(), "Some verticals were not assigned to a group."

## Split temporal

Mesmo esquema testado no notebook 1: um mês de warmup (para as features de histórico), gaps de um mês entre as bases para simular a maturação do chargeback.

In [4]:
df["ym"] = df["created_at"].dt.strftime("%Y-%m")
conds = [
    df.ym < "2019-02",
    df.ym.between("2019-02", "2020-04"),
    df.ym == "2020-05",
    df.ym.between("2020-06", "2020-08"),
    df.ym == "2020-09",
    df.ym >= "2020-10",
]
df["split"] = np.select(
    conds,
    ["warmup", "train", "gap1", "valid", "gap2", "test"],
    default="unassigned",
)
df.groupby("split")["is_fraud"].agg(["size", "sum", "mean"])

,size,sum,mean
split,,,
gap1,13494,81,0.006003
gap2,12793,69,0.005394
test,51557,171,0.003317
train,203934,1159,0.005683
valid,48135,203,0.004217
warmup,9694,99,0.010213


In [5]:
target = "is_fraud"
features = ["vertical_group"]

train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
test = df[df["split"] == "test"]

## WoE

O WoE é calculado **somente na base de treino** e reaplicado em validação e teste.

In [6]:
encoder = WoEEncoder(features, target=target).fit(train)

X_train = encoder.transform(train)
X_valid = encoder.transform(valid)
X_test = encoder.transform(test)

encoder.tables_["vertical_group"]

is_fraud,n_legit,n_fraud,n_txns,pct_population,fraud_rate,lift,WOE,IV
vertical_group,,,,,,,,
vertical_high_risk,45020,675,45695,0.224068,0.014772,2.599210,0.963833,0.347034
vertical_medium_risk,57022,278,57300,0.280973,0.004852,0.853682,-0.158530,0.006536
vertical_low_risk,100733,206,100939,0.494959,0.002041,0.359099,-1.026680,0.327337


In [7]:
encoder.iv()

vertical_group    0.680907
Name: IV, dtype: float64

## Regressão logística

In [8]:
model = LogisticRegression()
model.fit(X_train, train[target])

pd.Series(
    np.append(model.intercept_, model.coef_.ravel()),
    index=["intercept", *X_train.columns],
    name="coef",
)

intercept            -5.160607
vertical_group_woe    0.994343
Name: coef, dtype: float64

## Métricas

PR AUC (average precision), ROC AUC e KS em cada base.

In [9]:
baseline_metrics = metrics_by_split({
    "train": (train[target], model.predict_proba(X_train)[:, 1]),
    "valid": (valid[target], model.predict_proba(X_valid)[:, 1]),
    "test": (test[target], model.predict_proba(X_test)[:, 1]),
})

baseline_metrics

,n_txns,fraud_rate,pr_auc,roc_auc,ks
train,203934.0,0.005683,0.011833,0.714777,0.360379
valid,48135.0,0.004217,0.007781,0.681241,0.304716
test,51557.0,0.003317,0.006615,0.709953,0.329356
